In [1]:
import os
import json

root = "/Users/shaury/Downloads/metrics/"

ls = os.listdir(root)

def json_read(path):
    with open(os.path.join(root, path)) as f:
        return json.loads(f.read())

In [2]:
f = json_read(ls[0])

In [3]:
f['overall_metrics']

{'REPORT_GENERATION': {'bleu4': {'mean': 13.326444906979518},
  'bleu2': {'mean': 26.480437707055675},
  'rouge1': {'mean': 42.93877588630506},
  'rouge2': {'mean': 18.12182532771883},
  'rougeL': {'mean': 34.23420826560548},
  'bertscore': {'mean': 51.25361348718705},
  'f1-score': {'mean': 52.76655036678031},
  'precision': {'mean': 61.84966897903126},
  'recall': {'mean': 46.00966329077198},
  'meteor': {'mean': 38.838236342198776},
  'chexbert-5_micro avg_f1-score': {'mean': 0.0},
  'chexbert-all_micro avg_f1-score': {'mean': 0.32019704433497537},
  'chexbert-5_macro avg_f1-score': {'mean': 0.0},
  'chexbert-all_macro avg_f1-score': {'mean': 0.047674807627969225},
  'radgraph': {'mean': 0.33958632855302395},
  'green': {'mean': 0.47196186249216554}}}

In [17]:
from collections import defaultdict

metrics = defaultdict(lambda : [])

for p in ls:
    f = json_read(p)
    metric = f['overall_metrics']['REPORT_GENERATION']
    metrics['name'].append(p)

    for m in metric:
        metrics[m].append(metric[m]['mean'])

In [20]:
import pandas as pd

pd.DataFrame(metrics).to_csv("./metrics.csv")

In [1]:
from utils import XRayReportDataset

In [2]:
dataset = XRayReportDataset("../eden/intermediate/test.csv")

In [3]:
dataset[0:5]

{'messages': [{'role': 'user',
   'content': [{'type': 'text',
     'text': 'Given the indication:\n\nINDICATION : 0    XXXX year old smoking on oxygen and nasal cann...\n1    V76.12 SCREENING MAMMOGRAM XXXX,no hx ca or im...\n2                         Hematemesis; BACK PAIN 724.5\n3                       XXXX-year-old, MVA, chest pain\n4                     XXXX-year-old female, XXXX bleed\nName: indication, dtype: str\n\nPerform a systematic review of this chest X-ray against the following clinical categories:\n1. Cardiomegaly & Cardiovascular\n2. Alveolar Opacities & Infections\n3. Pleural Space Diseases\n4. COPD & Hyperinflation\n5. Atelectasis & Volume Loss\n6. Nodules, Masses & Fibrosis\n7. Bones & Spine\n8. Medical Devices\n9. Diaphragm & Extrathoracic\n\nBased on this systematic review, generate the findings and impression in the xml format as specified below.\n\noutput format:\n\n<findings> Findings </findings>\n<impression> Impression </impression>\n'}]},
  {'role': 'assistan

In [5]:
dataset[0:5]

{'messages': [{'role': 'user',
   'content': [{'type': 'text',
     'text': 'Given the indication:\n\nINDICATION : 0    XXXX year old smoking on oxygen and nasal cann...\n1    V76.12 SCREENING MAMMOGRAM XXXX,no hx ca or im...\n2                         Hematemesis; BACK PAIN 724.5\n3                       XXXX-year-old, MVA, chest pain\n4                     XXXX-year-old female, XXXX bleed\nName: indication, dtype: str\n\nPerform a systematic review of this chest X-ray against the following clinical categories:\n1. Cardiomegaly & Cardiovascular\n2. Alveolar Opacities & Infections\n3. Pleural Space Diseases\n4. COPD & Hyperinflation\n5. Atelectasis & Volume Loss\n6. Nodules, Masses & Fibrosis\n7. Bones & Spine\n8. Medical Devices\n9. Diaphragm & Extrathoracic\n\nBased on this systematic review, generate the findings and impression in the xml format as specified below.\n\noutput format:\n\n<findings> Findings </findings>\n<impression> Impression </impression>\n'}]},
  {'role': 'assistan

In [7]:
total_len = 198
BATCH_SIZE = 8

In [8]:
i = 194

In [9]:
batch_df = [dataset[idx] for idx in range(i, min(BATCH_SIZE+i, total_len))]

In [16]:
def parse_images(message):
    content = message[0]['content']

    images = []

    for item in content:
        if item['type'] == "image":
            images.append(item['image'])
    return images

In [18]:
parse_images(batch_df[0]['messages'])

[<PIL.Image.Image image mode=RGB size=2048x2496>,
 <PIL.Image.Image image mode=RGB size=2048x2496>]

In [19]:
from transformers import AutoProcessor
BASE_MODEL_ID="Qwen/Qwen3-VL-2B-Instruct"
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)
processor.tokenizer.padding_side = "left"

In [23]:
processor.tokenizer.padding_side = "left"

In [24]:
batch_df = [dataset[idx] for idx in range(i, min(BATCH_SIZE+i, total_len))]


texts = []
batch_images = []

for row in batch_df:
    # Apply chat template for this specific row
    messages = row["messages"]
    batch_images.extend(parse_images(messages))
    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    texts.append(text_prompt)

# Tokenize the entire batch at once
inputs = processor(
    text=texts,
    images=batch_images if len(batch_images) > 0 else None,
    return_tensors="pt",
    padding=True,
)

In [25]:
inputs

{'input_ids': tensor([[151643, 151643, 151643,  ..., 151644,  77091,    198],
        [151644,    872,    198,  ..., 151644,  77091,    198],
        [151643, 151643, 151643,  ..., 151644,  77091,    198],
        [151643, 151643, 151643,  ..., 151644,  77091,    198]]), 'attention_mask': tensor([[0, 0, 0,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1]]), 'pixel_values': tensor([[ 0.0353,  0.0118,  0.0275,  ..., -0.4039, -0.4118, -0.4275],
        [-0.1529, -0.1451, -0.1529,  ..., -0.4980, -0.5059, -0.5059],
        [-0.2392, -0.2627, -0.2549,  ..., -0.4275, -0.4275, -0.4431],
        ...,
        [-0.3569, -0.3490, -0.3569,  ..., -0.3490, -0.3412, -0.3412],
        [-0.4196, -0.4196, -0.3961,  ..., -0.1608, -0.1686, -0.1765],
        [-0.3804, -0.3804, -0.3804,  ..., -0.1529, -0.1451, -0.1373]]), 'image_grid_thw': tensor([[  1, 156, 128],
        [  1, 156, 128],
        [  1, 128, 156],
        [  1, 156, 128],
   